# Manufacturing Data Analysis EDA Skeleton

## Project Context

This notebook is a minimal starting point for later exploratory data analysis and project presentation work.

Current goal:

- understand equipment, line, shift, quality, and failure behavior from the current processed manufacturing dataset
- keep the analysis aligned with the current repository structure and contract tests

Important boundary:

- several KPI fields in this project, including `planned_production`, `actual_production`, `defect_rate`, `availability`, `performance`, `quality_rate`, and `oee`, are current project proxy / simulated metrics
- this notebook should support structured analysis, but not claim industrial-grade plant conclusions at this stage


## Analysis Questions

This notebook is designed to help answer questions like:

1. Which equipment or production lines show weaker OEE performance?
2. Do `Day`, `Evening`, and `Night` shifts show different KPI patterns?
3. How does machine failure relate to quality and OEE behavior?
4. Which equipment appears more often in failure-labelled records?
5. Which insights are supported by current proxy metrics, and which conclusions are still outside the scope of this project?


In [13]:
from pathlib import Path

import pandas as pd

## Data Loading

Choose one processed CSV source.

- `manufacturing_data_processed.csv` is the current legacy reference
- `data/processed/manufacturing_data_processed_refactor.csv` is the current refactor snapshot

The default below uses the legacy processed CSV.

In [14]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

legacy_processed_path = PROJECT_ROOT / 'manufacturing_data_processed.csv'
refactor_processed_path = PROJECT_ROOT / 'data' / 'processed' / 'manufacturing_data_processed_refactor.csv'

# Change this path if you want to inspect the refactor snapshot instead.
data_path = legacy_processed_path

print(f'Using dataset: {data_path}')
df = pd.read_csv(data_path)

print(f'Shape: {df.shape}')
display(df.head())

Using dataset: C:\Itmes_2\manufacturing_data_processed.csv
Shape: (10000, 26)


,UDI,production_time,equipment_id,production_line,shift,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],...,planned_production,actual_production,defect_rate,defect_count,qualified_count,availability,performance,quality_rate,oee,Machine failure
0,1,2025-01-01 08:00:00,Line2_EQ01,Line2,Day,M,298.1,308.6,1551,42.8,...,1.25,1.250000,0.064156,0,1.250000,1.0,1.000000,1.0,1.000000,0
1,2,2025-01-01 08:15:00,Line1_EQ01,Line1,Day,L,298.2,308.7,1408,46.3,...,1.00,0.936354,0.122071,0,0.936354,1.0,0.936354,1.0,0.936354,0
2,3,2025-01-01 08:30:00,Line1_EQ02,Line1,Day,L,298.1,308.5,1498,49.4,...,1.00,0.985284,0.174407,0,0.985284,1.0,0.985284,1.0,0.985284,0
3,4,2025-01-01 08:45:00,Line1_EQ03,Line1,Day,L,298.2,308.6,1433,39.5,...,1.00,0.936163,0.025151,0,0.936163,1.0,0.936163,1.0,0.936163,0
4,5,2025-01-01 09:00:00,Line1_EQ01,Line1,Day,L,298.2,308.7,1408,40.0,...,1.00,0.899056,0.017037,0,0.899056,1.0,0.899056,1.0,0.899056,0


In [15]:
df.columns.tolist()

['UDI',
 'production_time',
 'equipment_id',
 'production_line',
 'shift',
 'Type',
 'Air temperature [K]',
 'Process temperature [K]',
 'Rotational speed [rpm]',
 'Torque [Nm]',
 'Tool wear [min]',
 'process_stability_score',
 'air_temp_dev',
 'process_temp_dev',
 'torque_dev',
 'theoretical_cycle_time',
 'planned_production',
 'actual_production',
 'defect_rate',
 'defect_count',
 'qualified_count',
 'availability',
 'performance',
 'quality_rate',
 'oee',
 'Machine failure']

## Dataset Overview

Start with the most basic checks before writing any interpretation.

This section answers a simple question first:

- What data do we actually have in hand before we compare equipment, lines, shifts, or failures?


### A. Basic Dataset Size And Key Column Presence

This check confirms whether the dataset has the expected scale and whether the main analysis fields are present.

In [16]:
key_columns = ['production_time', 'equipment_id', 'oee', 'Machine failure']

basic_overview = {
    'row_count': len(df),
    'column_count': len(df.columns),
    'equipment_count': df['equipment_id'].nunique(),
    'production_line_count': df['production_line'].nunique(),
    'shift_categories': sorted(df['shift'].dropna().unique().tolist()),
    'key_columns_present': {column: (column in df.columns) for column in key_columns},
}

basic_overview

{'row_count': 10000,
 'column_count': 26,
 'equipment_count': 9,
 'production_line_count': 3,
 'shift_categories': ['Day', 'Evening', 'Night'],
 'key_columns_present': {'production_time': True,
  'equipment_id': True,
  'oee': True,
  'Machine failure': True}}

### B. Time Range Overview

This check tells us how wide the current analysis window is and how many calendar dates are covered.

In [17]:
production_time_series = pd.to_datetime(df['production_time'])

time_overview = {
    'min_production_time': production_time_series.min(),
    'max_production_time': production_time_series.max(),
    'date_count': production_time_series.dt.date.nunique(),
}

time_overview

{'min_production_time': Timestamp('2025-01-01 08:00:00'),
 'max_production_time': Timestamp('2025-04-15 11:45:00'),
 'date_count': 105}

### C. Basic Missing Check

This is not a full data-quality audit. It only checks whether the most important fields are missing obvious values.

In [18]:
missing_check_columns = [
    'production_time',
    'equipment_id',
    'production_line',
    'shift',
    'oee',
    'availability',
    'performance',
    'quality_rate',
    'Machine failure',
]

missing_summary = df[missing_check_columns].isna().sum().rename('missing_count').to_frame()
missing_summary['missing_ratio'] = (missing_summary['missing_count'] / len(df)).round(6)
missing_summary

,missing_count,missing_ratio
production_time,0,0.0
equipment_id,0,0.0
production_line,0,0.0
shift,0,0.0
oee,0,0.0
availability,0,0.0
performance,0,0.0
quality_rate,0,0.0
Machine failure,0,0.0


### D. Basic KPI Summary

This gives a first numeric profile of the current KPI fields.

Important reminder:

- these are current project proxy / simulated metrics
- treat this as a local analysis baseline, not as industrial KPI certification


In [19]:
kpi_summary = df[['oee', 'availability', 'performance', 'quality_rate']].describe().T
kpi_summary

,count,mean,std,min,25%,50%,75%,max
oee,10000.0,0.927615,0.179820,0.000000,0.922752,0.975911,1.0,1.0
availability,10000.0,0.966100,0.180981,0.000000,1.000000,1.000000,1.0,1.0
performance,10000.0,0.958071,0.048734,0.748838,0.924545,0.976743,1.0,1.0
quality_rate,10000.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.0,1.0


### E. Short Interpretation Notes

Use this area to write 2 to 4 short observations after you inspect the outputs above.

Suggested prompts:

- Does the dataset size match what you expected?
- Are the key analysis columns present and mostly complete?
- Does the current time range look reasonable for a first analysis pass?
- Do the KPI summaries look broadly plausible under the current proxy logic?


## Equipment / Line KPI Analysis

Use this section to compare equipment and production lines.

This section answers the next practical question:

- Which equipment or lines look weaker under the current KPI rules, and where should deeper analysis start?


### A. Equipment-Level KPI Summary

This table is the first place to look for weaker equipment under the current proxy KPI design.

Suggested reading habit:

- first scan low `avg_oee`
- then check whether low `avg_oee` appears together with higher `failure_record_count` or `total_defect_count`


In [20]:
equipment_kpi = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        production_line=('production_line', 'first'),
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count', 'total_defect_count'], ascending=[True, False, False])
    .reset_index(drop=True)
)

equipment_kpi.head(10)

,equipment_id,production_line,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_actual_production,total_defect_count,failure_record_count
0,Line1_EQ03,Line1,0.918856,0.956500,0.957912,1.0,1915.823164,0,87
1,Line1_EQ01,Line1,0.921774,0.960500,0.957126,1.0,1914.251794,0,79
2,Line1_EQ02,Line1,0.928590,0.965500,0.959432,1.0,1918.864141,0,69
3,Line2_EQ03,Line2,0.928958,0.967968,0.958314,1.0,1196.694055,0,32
4,Line2_EQ01,Line2,0.930808,0.969970,0.957812,1.0,1196.067625,0,30
5,Line3_EQ02,Line3,0.932230,0.970060,0.958895,1.0,480.406193,0,10
6,Line2_EQ02,Line2,0.937823,0.978979,0.956685,1.0,1194.660757,0,21
7,Line3_EQ01,Line3,0.941632,0.982090,0.958246,1.0,481.518568,0,6
8,Line3_EQ03,Line3,0.946444,0.985030,0.959737,1.0,480.828258,0,5


Interpretation prompt:

- Which equipment sits near the bottom of `avg_oee`?
- Are those same equipment IDs also showing more failure-labelled records or more total defects?
- Under the current proxy KPI rules, this supports prioritizing inspection targets, but it does not prove root cause.


### B. Production-Line KPI Summary

This table rolls the same logic up to production-line level so you can see whether weaker equipment also cluster into weaker lines.

In [21]:
line_kpi = (
    df.groupby('production_line', as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_actual_production=('actual_production', 'sum'),
        total_defect_count=('defect_count', 'sum'),
        failure_record_count=('Machine failure', 'sum'),
    )
    .sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])
    .reset_index(drop=True)
)

line_kpi

,production_line,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_actual_production,total_defect_count,failure_record_count
0,Line1,0.923073,0.960833,0.958157,1.0,5748.939100,0,235
1,Line2,0.932530,0.972306,0.957604,1.0,3587.422437,0,83
2,Line3,0.940103,0.979063,0.958958,1.0,1442.753019,0,21


Interpretation prompt:

- Which production line looks weakest on average `oee`?
- Does the weaker line also show more failure-labelled records or more defects?
- Under the current project scope, this supports line-level comparison, but not final industrial performance judgment.


### C. Simple Ranking Views

These quick ranking tables help you identify where to focus next without introducing charts yet.

In [22]:
lowest_avg_oee_equipment = equipment_kpi.nsmallest(5, 'avg_oee')
highest_defect_equipment = equipment_kpi.nlargest(5, 'total_defect_count')
line_ranking = line_kpi.sort_values(['avg_oee', 'failure_record_count'], ascending=[True, False])

display(lowest_avg_oee_equipment)
display(highest_defect_equipment)
display(line_ranking)

,equipment_id,production_line,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_actual_production,total_defect_count,failure_record_count
0,Line1_EQ03,Line1,0.918856,0.956500,0.957912,1.0,1915.823164,0,87
1,Line1_EQ01,Line1,0.921774,0.960500,0.957126,1.0,1914.251794,0,79
2,Line1_EQ02,Line1,0.928590,0.965500,0.959432,1.0,1918.864141,0,69
3,Line2_EQ03,Line2,0.928958,0.967968,0.958314,1.0,1196.694055,0,32
4,Line2_EQ01,Line2,0.930808,0.969970,0.957812,1.0,1196.067625,0,30


,equipment_id,production_line,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_actual_production,total_defect_count,failure_record_count
0,Line1_EQ03,Line1,0.918856,0.956500,0.957912,1.0,1915.823164,0,87
1,Line1_EQ01,Line1,0.921774,0.960500,0.957126,1.0,1914.251794,0,79
2,Line1_EQ02,Line1,0.928590,0.965500,0.959432,1.0,1918.864141,0,69
3,Line2_EQ03,Line2,0.928958,0.967968,0.958314,1.0,1196.694055,0,32
4,Line2_EQ01,Line2,0.930808,0.969970,0.957812,1.0,1196.067625,0,30


,production_line,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_actual_production,total_defect_count,failure_record_count
0,Line1,0.923073,0.960833,0.958157,1.0,5748.939100,0,235
1,Line2,0.932530,0.972306,0.957604,1.0,3587.422437,0,83
2,Line3,0.940103,0.979063,0.958958,1.0,1442.753019,0,21


Interpretation prompt:

- Do the weakest `avg_oee` equipment match the highest-defect equipment?
- Are the same lines repeatedly appearing at the weaker end of the ranking?
- These rankings are good starting points for later shift and failure analysis, but they are still descriptive, not causal.


## Shift KPI Analysis

Use this section to compare `Day`, `Evening`, and `Night` shifts.

Suggested focus:

- average `oee`
- average `availability`
- average `performance`
- average `quality_rate`
- total `defect_count`
- failure-labelled record count


In [23]:
shift_kpi = (
    df.groupby(['production_line', 'shift'], as_index=False)
    .agg(
        avg_oee=('oee', 'mean'),
        avg_availability=('availability', 'mean'),
        avg_performance=('performance', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
        total_defect_count=('defect_count', 'sum'),
        failure_records=('Machine failure', 'sum'),
    )
    .sort_values(['production_line', 'shift'])
)

shift_kpi

,production_line,shift,avg_oee,avg_availability,avg_performance,avg_quality_rate,total_defect_count,failure_records
0,Line1,Day,0.927460,0.964535,0.959187,1.0,0,71
1,Line1,Evening,0.918243,0.956522,0.957319,1.0,0,88
2,Line1,Night,0.923576,0.961499,0.957970,1.0,0,76
3,Line2,Day,0.932356,0.971859,0.957886,1.0,0,28
4,Line2,Evening,0.931324,0.971458,0.957486,1.0,0,28
5,Line2,Night,0.933857,0.973555,0.957441,1.0,0,27
6,Line3,Day,0.942867,0.982709,0.958862,1.0,0,6
7,Line3,Evening,0.938341,0.978328,0.957266,1.0,0,7
8,Line3,Night,0.938933,0.975976,0.960701,1.0,0,8


## Failure Summary

Use this section to inspect failure-labelled records and compare them with non-failure records.

Suggested focus:

- number of failure-labelled records
- average `oee` under failure vs non-failure records
- average `quality_rate` under failure vs non-failure records
- equipment with more failure-labelled records


In [24]:
failure_summary = df.groupby('Machine failure', as_index=False).agg(
    record_count=('UDI', 'count'),
    avg_oee=('oee', 'mean'),
    avg_quality_rate=('quality_rate', 'mean'),
    avg_defect_rate=('defect_rate', 'mean'),
)

failure_by_equipment = (
    df.groupby('equipment_id', as_index=False)
    .agg(
        failure_records=('Machine failure', 'sum'),
        avg_oee=('oee', 'mean'),
        avg_quality_rate=('quality_rate', 'mean'),
    )
    .sort_values(['failure_records', 'avg_oee'], ascending=[False, True])
)

display(failure_summary)
display(failure_by_equipment.head(10))

,Machine failure,record_count,avg_oee,avg_quality_rate,avg_defect_rate
0,0,9661,0.960165,1.0,0.124925
1,1,339,0.000000,1.0,0.197400


,equipment_id,failure_records,avg_oee,avg_quality_rate
2,Line1_EQ03,87,0.918856,1.0
0,Line1_EQ01,79,0.921774,1.0
1,Line1_EQ02,69,0.928590,1.0
5,Line2_EQ03,32,0.928958,1.0
3,Line2_EQ01,30,0.930808,1.0
4,Line2_EQ02,21,0.937823,1.0
7,Line3_EQ02,10,0.932230,1.0
6,Line3_EQ01,6,0.941632,1.0
8,Line3_EQ03,5,0.946444,1.0


## Findings Draft

Use this section to draft short findings in plain language.

Finding 1

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 2

- Observation:
- Evidence:
- Business meaning:
- Limitation:

Finding 3

- Observation:
- Evidence:
- Business meaning:
- Limitation:


## Limitations

Keep these limits visible when writing conclusions:

- current OEE, quality, and production fields are proxy / simulated metrics under current project rules
- current timestamps, lines, and equipment identifiers are partly generated by the preprocessing logic
- this notebook supports structured local analysis, not industrial-grade operational claims
- later notebook and dashboard work should stay aligned with `docs/analysis_report_template.md`
